<a href="https://colab.research.google.com/github/gunagantinikhil/Major_11/blob/main/module_1_sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
%matplotlib inline

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

BASE = "/kaggle/input"

for root, dirs, files in os.walk(BASE):
    print(root, "| dirs:", len(dirs), "| files:", len(files))
    for f in files:
        print("    ", f)

csv_files = glob.glob(BASE + "/**/*.csv", recursive=True)
print("\nCSV files:", csv_files)

df = pd.read_csv(csv_files[0])
print("\nShape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nDtypes:\n", df.dtypes)
print("\nMemory (MB):", round(df.memory_usage(deep=True).sum() / 1024**2, 2))
display(df.head())

/kaggle/input | dirs: 0 | files: 0

CSV files: []


IndexError: list index out of range

In [ ]:
df = df.drop(columns=["Unnamed: 0"])

print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nMissing %:\n", (df.isnull().mean() * 100).round(3))
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate statements:", df["statement"].duplicated().sum())

print("\nClass distribution:\n", df["status"].value_counts())
print("\nClass distribution %:\n", (df["status"].value_counts(normalize=True) * 100).round(2))
print("\nUnique classes:", df["status"].nunique())

stmt = df["statement"].dropna().astype(str)
print("\nEmpty or whitespace-only statements:", (stmt.str.strip() == "").sum())
print("Non-string entries:", (~df["statement"].dropna().map(lambda x: isinstance(x, str))).sum())

print("\nSample per class:")
for c in df["status"].dropna().unique():
    print("\n[", c, "]")
    print(df.loc[df["status"] == c, "statement"].dropna().iloc[0][:200])

In [ ]:
before = len(df)
df = df.dropna(subset=["statement"]).reset_index(drop=True)
after_na = len(df)
df = df.drop_duplicates(subset=["statement"]).reset_index(drop=True)
after_dup = len(df)

print("Rows before:", before)
print("After dropping missing:", after_na, "| removed:", before - after_na)
print("After dropping duplicates:", after_dup, "| removed:", after_na - after_dup)

conflict = df.groupby("statement")["status"].nunique()
print("Statements with conflicting labels:", (conflict > 1).sum())

df["char_count"] = df["statement"].str.len()
df["word_count"] = df["statement"].str.split().str.len()

order = df["status"].value_counts().index

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.countplot(data=df, y="status", order=order, palette="viridis", ax=ax[0])
ax[0].set_title("Class Distribution (Counts)")
ax[0].set_xlabel("Count")
ax[0].set_ylabel("Status")
for p in ax[0].patches:
    ax[0].annotate(int(p.get_width()), (p.get_width(), p.get_y() + p.get_height() / 2),
                   ha="left", va="center", fontsize=9)

ax[1].pie(df["status"].value_counts()[order], labels=order, autopct="%1.1f%%",
          colors=sns.color_palette("viridis", len(order)), startangle=90)
ax[1].set_title("Class Distribution (Percentage)")
plt.tight_layout()
plt.show()

print("\nLength summary:")
print(df[["char_count", "word_count"]].describe().round(2))

In [ ]:
cap = df["word_count"].quantile(0.99)
plot_df = df[df["word_count"] <= cap]

fig, ax = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(plot_df["word_count"], bins=60, kde=True, color="teal", ax=ax[0])
ax[0].set_title("Word Count Distribution (up to 99th percentile)")
ax[0].set_xlabel("Words")
ax[0].set_ylabel("Frequency")

sns.boxplot(data=plot_df, x="status", y="word_count", order=order, palette="viridis", ax=ax[1])
ax[1].set_title("Word Count by Class")
ax[1].set_xlabel("Status")
ax[1].set_ylabel("Words")
ax[1].tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.show()

summary = df.groupby("status")[["word_count", "char_count"]].agg(["mean", "median", "max"]).round(1)
print(summary.loc[order])

print("\n99th percentile word count:", int(cap))
print("Statements under 5 words:", (df["word_count"] < 5).sum())
print("Statements over 500 words:", (df["word_count"] > 500).sum())
print("Short statements by class:")
print(df[df["word_count"] < 5]["status"].value_counts())

In [ ]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

df["has_url"] = df["statement"].str.contains(r"http\S+|www\.\S+", regex=True).astype(int)
df["has_mention"] = df["statement"].str.contains(r"[@/]u/\w+|@\w+", regex=True).astype(int)
df["exclaim_count"] = df["statement"].str.count("!")
df["question_count"] = df["statement"].str.count(r"\?")
df["upper_ratio"] = df["statement"].apply(lambda s: sum(c.isupper() for c in s) / max(len(s), 1))

def clean_text(s):
    s = s.lower()
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"[@/]u/\w+|@\w+", " ", s)
    s = re.sub(r"&\w+;", " ", s)
    s = re.sub(r"\*\*|\*|_|~|`|#|>", " ", s)
    s = re.sub(r"(.)\1{3,}", r"\1\1", s)
    s = re.sub(r"[^a-z\s']", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

stop = set(ENGLISH_STOP_WORDS) - {"not", "no", "never", "nothing", "nobody", "cannot", "nor"}

def remove_stop(s):
    return " ".join(w for w in s.split() if w not in stop and len(w) > 1)

df["clean_text"] = df["statement"].apply(clean_text)
df["clean_nostop"] = df["clean_text"].apply(remove_stop)

empty_mask = df["clean_nostop"].str.strip() == ""
print("Rows empty after cleaning:", empty_mask.sum())
print(df.loc[empty_mask, "status"].value_counts())

df = df.loc[~empty_mask].reset_index(drop=True)
df["clean_word_count"] = df["clean_nostop"].str.split().str.len()

print("\nRows remaining:", len(df))
print("\nBefore vs after (3 samples):")
for i in df.sample(3, random_state=42).index:
    print("\nORIGINAL:", df.loc[i, "statement"][:250])
    print("CLEAN   :", df.loc[i, "clean_text"][:250])
    print("NOSTOP  :", df.loc[i, "clean_nostop"][:250])

print("\nFlag counts:")
print(df[["has_url", "has_mention"]].sum())
print("\nClean word count summary:")
print(df["clean_word_count"].describe().round(2))

NameError: name 'df' is not defined

In [ ]:
from collections import Counter

classes = list(order)
top_n = 12

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()
palette = sns.color_palette("viridis", len(classes))

for i, c in enumerate(classes):
    words = " ".join(df.loc[df["status"] == c, "clean_nostop"]).split()
    common = Counter(words).most_common(top_n)
    w, n = zip(*common)
    sns.barplot(x=list(n), y=list(w), color=palette[i], ax=axes[i])
    axes[i].set_title(c)
    axes[i].set_xlabel("Frequency")

axes[-1].axis("off")
plt.suptitle("Top 12 Words per Class", fontsize=16)
plt.tight_layout()
plt.show()

global_counts = Counter(" ".join(df["clean_nostop"]).split())
print("Vocabulary size:", len(global_counts))
print("Words appearing only once:", sum(1 for v in global_counts.values() if v == 1))
print("Words appearing 5+ times:", sum(1 for v in global_counts.values() if v >= 5))
print("\nTop 20 overall:", global_counts.most_common(20))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=8000, ngram_range=(1, 2), min_df=5, max_df=0.5, sublinear_tf=True)
X = tfidf.fit_transform(df["clean_nostop"])
terms = np.array(tfidf.get_feature_names_out())
print("TF-IDF matrix:", X.shape)

class_means = {}
for c in classes:
    idx = np.where(df["status"].values == c)[0]
    class_means[c] = np.asarray(X[idx].mean(axis=0)).ravel()

mean_df = pd.DataFrame(class_means, index=terms)
overall = mean_df.mean(axis=1)
distinct = mean_df.sub(overall, axis=0)

selected = []
for c in classes:
    top = distinct[c].sort_values(ascending=False).head(6).index.tolist()
    print("\n" + c + ":", top)
    selected.extend(top)
selected = list(dict.fromkeys(selected))

plt.figure(figsize=(12, 14))
sns.heatmap(mean_df.loc[selected, classes], cmap="YlGnBu", annot=False, linewidths=0.3)
plt.title("Mean TF-IDF of Class-Distinctive Terms")
plt.xlabel("Status")
plt.ylabel("Term")
plt.tight_layout()
plt.show()

In [ ]:
meta_cols = ["char_count", "word_count", "clean_word_count", "exclaim_count", "question_count", "upper_ratio", "has_url", "has_mention"]

profile = df.groupby("status")[meta_cols].mean().loc[classes].round(3)
print(profile)

fig, ax = plt.subplots(1, 2, figsize=(18, 6))

scaled = (profile - profile.min()) / (profile.max() - profile.min())
sns.heatmap(scaled, annot=profile, fmt=".2f", cmap="coolwarm", linewidths=0.5, ax=ax[0], cbar_kws={"label": "Min-max scaled"})
ax[0].set_title("Mean Meta-Features by Class")
ax[0].set_xlabel("Feature")
ax[0].set_ylabel("Status")
ax[0].tick_params(axis="x", rotation=40)

sns.heatmap(df[meta_cols].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0, linewidths=0.5, ax=ax[1])
ax[1].set_title("Meta-Feature Correlation")
ax[1].tick_params(axis="x", rotation=40)

plt.tight_layout()
plt.show()

print("\nShare of statements with URL by class (%):")
print((df.groupby("status")["has_url"].mean().loc[classes] * 100).round(2))
print("\nShare of statements with exclamation marks by class (%):")
print((df.groupby("status")["exclaim_count"].apply(lambda s: (s > 0).mean()).loc[classes] * 100).round(2))

In [ ]:
import sys
import subprocess

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vaderSentiment"])
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()
df["compound"] = df["statement"].str.slice(0, 3000).apply(lambda s: sia.polarity_scores(s)["compound"])
df["polarity"] = pd.cut(df["compound"], bins=[-1.01, -0.05, 0.05, 1.01], labels=["Negative", "Neutral", "Positive"])

print(df.groupby("status")["compound"].agg(["mean", "median", "std"]).loc[classes].round(3))

pol_share = pd.crosstab(df["status"], df["polarity"], normalize="index").loc[classes] * 100
print("\nPolarity share by class (%):")
print(pol_share.round(2))

fig, ax = plt.subplots(1, 2, figsize=(18, 6))
sns.boxplot(data=df, x="status", y="compound", order=classes, palette="viridis", ax=ax[0])
ax[0].set_title("VADER Compound Score by Class")
ax[0].set_xlabel("Status")
ax[0].set_ylabel("Compound score")
ax[0].tick_params(axis="x", rotation=40)

pol_share.plot(kind="bar", stacked=True, color=["#d62728", "#7f7f7f", "#2ca02c"], ax=ax[1])
ax[1].set_title("Polarity Composition by Class")
ax[1].set_xlabel("Status")
ax[1].set_ylabel("Percentage")
ax[1].tick_params(axis="x", rotation=40)
ax[1].legend(title="Polarity")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

le = LabelEncoder()
df["label"] = le.fit_transform(df["status"])
label_map = dict(zip(le.classes_, le.transform(le.classes_)))
print("Label mapping:", label_map)

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42)

print("\nTrain:", train_df.shape, "| Validation:", val_df.shape, "| Test:", test_df.shape)

split_counts = pd.DataFrame({
    "Train": train_df["status"].value_counts(),
    "Validation": val_df["status"].value_counts(),
    "Test": test_df["status"].value_counts()
}).loc[classes]
print("\nSplit counts:\n", split_counts)

split_pct = (split_counts / split_counts.sum() * 100).round(2)
print("\nSplit class share (%):\n", split_pct)

fig, ax = plt.subplots(1, 2, figsize=(18, 6))
split_counts.plot(kind="bar", ax=ax[0], colormap="viridis")
ax[0].set_title("Class Counts per Split")
ax[0].set_xlabel("Status")
ax[0].set_ylabel("Count")
ax[0].tick_params(axis="x", rotation=40)

split_pct.plot(kind="bar", ax=ax[1], colormap="viridis")
ax[1].set_title("Class Share per Split (%)")
ax[1].set_xlabel("Status")
ax[1].set_ylabel("Percentage")
ax[1].tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.show()

weights = len(train_df) / (len(classes) * train_df["label"].value_counts().sort_index())
class_weights = dict(zip(le.classes_, weights.round(3)))
print("\nSuggested class weights (inverse frequency):", class_weights)

In [ ]:
raw_rows = 53043
steps = pd.Series({
    "Raw": raw_rows,
    "After missing removal": 52681,
    "After duplicate removal": 51073,
    "After empty-clean removal": len(df)
})
print(steps)
print("\nTotal rows removed:", raw_rows - len(df), "(", round((raw_rows - len(df)) / raw_rows * 100, 2), "% )")

keep_cols = ["statement", "clean_text", "clean_nostop", "status", "label", "char_count", "word_count",
             "clean_word_count", "exclaim_count", "question_count", "upper_ratio", "has_url", "has_mention", "compound"]

print("\nFinal columns:", keep_cols)
print("Nulls in final data:", int(df[keep_cols].isnull().sum().sum()))
print("Duplicate statements remaining:", int(df["statement"].duplicated().sum()))
print("Final shape:", df[keep_cols].shape)

out_dir = "/kaggle/working"
train_df[keep_cols].to_csv(out_dir + "/train.csv", index=False)
val_df[keep_cols].to_csv(out_dir + "/val.csv", index=False)
test_df[keep_cols].to_csv(out_dir + "/test.csv", index=False)
print("\nSaved:", sorted(f for f in os.listdir(out_dir) if f.endswith(".csv")))

fig, ax = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(x=steps.index, y=steps.values, palette="viridis", ax=ax[0])
ax[0].set_title("Row Count Through Preprocessing")
ax[0].set_xlabel("Stage")
ax[0].set_ylabel("Rows")
ax[0].tick_params(axis="x", rotation=20)
for p in ax[0].patches:
    ax[0].annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom")

sns.histplot(df["clean_word_count"].clip(upper=df["clean_word_count"].quantile(0.99)), bins=50, kde=True, color="teal", ax=ax[1])
ax[1].set_title("Final Cleaned Word Count Distribution")
ax[1].set_xlabel("Words after cleaning")
ax[1].set_ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
n = len(df)
cc = df["status"].value_counts()
wc = df.groupby("status")["word_count"].median()
cw = df.groupby("status")["compound"].mean()
neg = (pd.crosstab(df["status"], df["polarity"], normalize="index")["Negative"] * 100)

line = "=" * 70
print(line)
print("MODULE 1 - EDA SUMMARY: MENTAL HEALTH SENTIMENT DATASET")
print(line)

print("\n1. DATASET")
print(f"   - Source file: Combined Data.csv, raw rows {raw_rows}, final rows {n}")
print(f"   - Rows removed: {raw_rows - n} ({(raw_rows - n) / raw_rows * 100:.2f}%)")
print("   - Target: status with 7 classes")

print("\n2. DATA QUALITY")
print("   - 362 missing statements removed")
print("   - 1608 duplicate statements removed, no conflicting labels found")
print("   - 123 rows empty after cleaning removed, mostly Normal")

print("\n3. CLASS BALANCE")
print(f"   - Largest: {cc.index[0]} ({cc.iloc[0] / n * 100:.1f}%), smallest: {cc.index[-1]} ({cc.iloc[-1] / n * 100:.1f}%)")
print(f"   - Imbalance ratio: {cc.iloc[0] / cc.iloc[-1]:.1f} to 1")
print("   - Stratified 70/15/15 split keeps class shares identical across splits")
print("   - Class weights are suggested for training")

print("\n4. TEXT LENGTH")
print(f"   - Median words: Normal {int(wc['Normal'])}, other classes {int(wc.drop('Normal').min())} to {int(wc.drop('Normal').max())}")
print("   - Normal posts are short, clinical-condition posts are long")
print(f"   - 99th percentile length: {int(df['word_count'].quantile(0.99))} words, useful for max sequence length")

print("\n5. VOCABULARY AND LANGUAGE")
print("   - Each class has clear marker words (anxiety, bipolar, manic, die, kill, avpd)")
print("   - Depression and Suicidal share many terms, so they are the hardest pair to separate")
print("   - Negation words were kept during stopword removal")

print("\n6. SENTIMENT")
print(f"   - Most negative class: {cw.idxmin()} (mean compound {cw.min():.2f})")
print(f"   - Only positive-leaning class: {cw.idxmax()} (mean compound {cw.max():.2f})")
print(f"   - Share of negative posts: highest {neg.idxmax()} ({neg.max():.1f}%), lowest {neg.idxmin()} ({neg.min():.1f}%)")

print("\n7. READINESS FOR NEXT MODULE")
print("   - Cleaned columns: clean_text, clean_nostop")
print("   - Label encoding done, train.csv, val.csv, test.csv saved")
print("   - Recommended: class weights or resampling, and macro F1 as the main metric")
print(line)

fig, ax = plt.subplots(1, 3, figsize=(20, 5))
sns.barplot(x=cc.index, y=cc.values, palette="viridis", ax=ax[0])
ax[0].set_title("Class Counts")
ax[0].set_xlabel("Status")
ax[0].set_ylabel("Count")
ax[0].tick_params(axis="x", rotation=40)

sns.barplot(x=wc.loc[classes].index, y=wc.loc[classes].values, palette="viridis", ax=ax[1])
ax[1].set_title("Median Word Count")
ax[1].set_xlabel("Status")
ax[1].set_ylabel("Words")
ax[1].tick_params(axis="x", rotation=40)

sns.barplot(x=cw.loc[classes].index, y=cw.loc[classes].values, palette="coolwarm_r", ax=ax[2])
ax[2].set_title("Mean VADER Compound Score")
ax[2].set_xlabel("Status")
ax[2].set_ylabel("Compound")
ax[2].tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.show()